# Uplift Random Forest

This notebook implements CausalML's `UpliftRandomForestClassifier` for the binary any-email-versus-no-email experiment. It is an uplift random forest, not Wager and Athey's honest causal forest. Model selection and overfitting evidence use out-of-fold predictions generated only within the 48,000-row training split; the 16,000-row test split is evaluated once at the end.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from causalml.inference.tree import UpliftRandomForestClassifier

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    FIGURES_DIR,
    MODEL_RESULTS_DIR,
    PREDICTIONS_DIR,
    PROCESSED_DATA_DIR,
    RANDOM_STATE,
    TABLES_DIR,
)
from src.metrics import (
    build_decile_table,
    compute_qini_auuc,
    stratified_uplift_folds,
)
from src.plotting import (
    plot_cumulative_gain_curve,
    plot_decile_lift_bar,
    plot_qini_curve,
    save_fig,
)

for directory in (FIGURES_DIR, MODEL_RESULTS_DIR, PREDICTIONS_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


Failed to import duecredit due to No module named 'duecredit'


## Load the shared train/test split


In [2]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")
treatment_train = pd.read_csv(
    PROCESSED_DATA_DIR / "treatment_train.csv"
).squeeze("columns")
treatment_test = pd.read_csv(
    PROCESSED_DATA_DIR / "treatment_test.csv"
).squeeze("columns")

assert list(X_train.columns) == list(X_test.columns)
assert len(X_train) == len(y_train) == len(treatment_train)
assert len(X_test) == len(y_test) == len(treatment_test)
assert set(treatment_train.unique()) <= {0, 1}
assert set(treatment_test.unique()) <= {0, 1}

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train treatment rate:", treatment_train.mean())


Train: (48000, 18) Test: (16000, 18)
Train treatment rate: 0.6670833333333334


## Training-only OOF evaluation

Each training row receives one uplift prediction from a forest fitted on the other four folds. Joint treatment/outcome stratification keeps all four experimental outcome cells represented despite the sparse visit outcome.


In [3]:
FOREST_PARAMETERS = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_leaf": 10,
    "random_state": RANDOM_STATE,
    "control_name": "control",
}
N_FOLDS = 5


def treatment_labels(values):
    return np.where(np.asarray(values) == 1, "treatment", "control")


def first_uplift_column(predictions):
    predictions = np.asarray(predictions)
    return predictions[:, 0] if predictions.ndim == 2 else predictions.ravel()


cf_oof = np.full(len(X_train), np.nan)
for fold_number, (fold_train_idx, fold_val_idx) in enumerate(
    stratified_uplift_folds(
        y_train,
        treatment_train,
        n_splits=N_FOLDS,
        random_state=RANDOM_STATE,
    ),
    start=1,
):
    fold_model = UpliftRandomForestClassifier(**FOREST_PARAMETERS)
    fold_model.fit(
        X_train.iloc[fold_train_idx].to_numpy(),
        treatment=treatment_labels(treatment_train.iloc[fold_train_idx]),
        y=y_train.iloc[fold_train_idx].to_numpy(),
    )
    cf_oof[fold_val_idx] = first_uplift_column(
        fold_model.predict(X_train.iloc[fold_val_idx].to_numpy())
    )
    print(f"Completed fold {fold_number}/{N_FOLDS}")

assert np.isfinite(cf_oof).all()
validation_scores = compute_qini_auuc(
    y_train,
    treatment_train,
    {"Uplift Random Forest": cf_oof},
    normalize=True,
)
print("OOF validation Qini:", validation_scores["qini_score"])
print("OOF validation AUUC:", validation_scores["auuc_score"])


Completed fold 1/5
Completed fold 2/5
Completed fold 3/5
Completed fold 4/5
Completed fold 5/5
OOF validation Qini: Uplift Random Forest    0.048572
dtype: float64
OOF validation AUUC: Uplift Random Forest    0.547398
dtype: float64


## Refit on all training data and evaluate once on test


In [4]:
forest = UpliftRandomForestClassifier(**FOREST_PARAMETERS)
forest.fit(
    X_train.to_numpy(),
    treatment=treatment_labels(treatment_train),
    y=y_train.to_numpy(),
)
cf_predicted_uplift = first_uplift_column(forest.predict(X_test.to_numpy()))

test_scores = compute_qini_auuc(
    y_test,
    treatment_test,
    {"Uplift Random Forest": cf_predicted_uplift},
    normalize=True,
)

metric_comparison = pd.DataFrame({
    "split": ["OOF validation", "Held-out test"],
    "qini_score": [
        float(validation_scores["qini_score"]["Uplift Random Forest"]),
        float(test_scores["qini_score"]["Uplift Random Forest"]),
    ],
    "auuc_score": [
        float(validation_scores["auuc_score"]["Uplift Random Forest"]),
        float(test_scores["auuc_score"]["Uplift Random Forest"]),
    ],
})
display(metric_comparison)


,split,qini_score,auuc_score
0,OOF validation,0.048572,0.547398
1,Held-out test,0.006527,0.507892


## Decile and curve diagnostics


In [5]:
decile_table = build_decile_table(
    y_test, treatment_test, cf_predicted_uplift, n_bins=10
)
display(decile_table)
decile_table.to_csv(TABLES_DIR / "causal_forest_decile_table.csv")

tau_test = {"Uplift Random Forest": cf_predicted_uplift}
fig_qini = plot_qini_curve(y_test, treatment_test, tau_test)
save_fig(fig_qini, "causal_forest_qini_curve.png")

fig_gain = plot_cumulative_gain_curve(y_test, treatment_test, tau_test)
save_fig(fig_gain, "causal_forest_gain_curve.png")

fig_decile = plot_decile_lift_bar(
    decile_table,
    title="Actual lift by predicted-uplift decile (Uplift Random Forest)",
)
save_fig(fig_decile, "causal_forest_decile_lift.png")


Figure(1000x800)
Figure(1000x800)
Figure(800x500)


,n_customers,n_treated,n_control,treated_outcome_rate,control_outcome_rate,actual_lift,actual_lift_se,actual_lift_ci_lower,actual_lift_ci_upper,mean_tau_hat
decile,,,,,,,,,,
1,1600,1059,541,0.208687,0.149723,0.058965,0.019795,0.020167,0.097763,0.128145
2,1600,1070,530,0.162617,0.105660,0.056956,0.017493,0.022670,0.091243,0.093906
3,1600,1081,519,0.154487,0.080925,0.073562,0.016264,0.041684,0.105440,0.083357
4,1600,1034,566,0.173114,0.107774,0.065340,0.017572,0.030900,0.099781,0.074036
5,1600,1070,530,0.175701,0.100000,0.075701,0.017482,0.041437,0.109965,0.064740
6,1600,1048,552,0.148855,0.090580,0.058275,0.016447,0.026039,0.090512,0.055962
7,1600,1091,509,0.145738,0.074656,0.071082,0.015818,0.040078,0.102085,0.047843
8,1600,1073,527,0.142591,0.117647,0.024944,0.017646,-0.009643,0.059531,0.039520
9,1600,1076,524,0.155204,0.108779,0.046426,0.017531,0.012065,0.080786,0.027245


PosixPath('/workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/figures/causal_forest_decile_lift.png')

## Model diagnostics


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(cf_predicted_uplift, bins=50, color="steelblue", alpha=0.8)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Test uplift distribution")
axes[0].set_xlabel("Predicted uplift")

metric_comparison.set_index("split")[["qini_score", "auuc_score"]].plot(
    kind="bar", ax=axes[1], rot=0
)
axes[1].set_title("OOF validation vs held-out test")
axes[1].set_ylabel("Normalized score")
plt.tight_layout()
save_fig(fig, "causal_forest_cv_results.png")

feature_importance = getattr(forest, "feature_importances_", None)
if feature_importance is not None:
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": np.asarray(feature_importance).ravel(),
    }).sort_values("importance", ascending=False)
    fig, ax = plt.subplots(figsize=(9, 6))
    top = importance_df.head(15).sort_values("importance")
    ax.barh(top["feature"], top["importance"], color="steelblue")
    ax.set_title("Uplift Random Forest feature importance")
    plt.tight_layout()
    save_fig(fig, "causal_forest_feature_importance.png")
else:
    importance_df = pd.DataFrame(columns=["feature", "importance"])
    print("This CausalML version did not expose feature_importances_.")


Figure(1300x500)
Figure(900x600)


## Save standardized predictions and results


In [7]:
test_predictions = pd.DataFrame({
    "test_row_id": np.arange(len(X_test)),
    "y_true": y_test.to_numpy(),
    "treatment": treatment_test.to_numpy(),
    "cf_predicted_uplift": cf_predicted_uplift,
})
oof_predictions = pd.DataFrame({
    "train_row_id": np.arange(len(X_train)),
    "y_true": y_train.to_numpy(),
    "treatment": treatment_train.to_numpy(),
    "cf_predicted_uplift": cf_oof,
})

test_path = PREDICTIONS_DIR / "causal_forest_predictions.csv"
oof_path = PREDICTIONS_DIR / "causal_forest_oof_predictions.csv"
test_predictions.to_csv(test_path, index=False)
oof_predictions.to_csv(oof_path, index=False)

results = {
    "metadata": {
        "model_type": "uplift_random_forest",
        "implementation": "causalml.inference.tree.UpliftRandomForestClassifier",
        "not_an_honest_causal_forest": True,
        "outcome": "visit",
        "treatment_definition": "Any email versus no email",
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "validation_method": f"{N_FOLDS}-fold OOF on X_train only",
    },
    "parameters": FOREST_PARAMETERS,
    "validation_qini_score": float(
        validation_scores["qini_score"]["Uplift Random Forest"]
    ),
    "validation_auuc_score": float(
        validation_scores["auuc_score"]["Uplift Random Forest"]
    ),
    "test_qini_score": float(test_scores["qini_score"]["Uplift Random Forest"]),
    "test_auuc_score": float(test_scores["auuc_score"]["Uplift Random Forest"]),
    "average_predicted_uplift": float(cf_predicted_uplift.mean()),
    "std_predicted_uplift": float(cf_predicted_uplift.std()),
    "top_features": importance_df.head(15).to_dict(orient="records"),
}

with open(MODEL_RESULTS_DIR / "causal_forest_results.json", "w") as file:
    json.dump(results, file, indent=4)

print("Saved test predictions:", test_path)
print("Saved OOF predictions:", oof_path)


Saved test predictions: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/predictions/causal_forest_predictions.csv
Saved OOF predictions: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/predictions/causal_forest_oof_predictions.csv


## Interpretation

The OOF scores are the model-selection evidence; the test scores are the final generalization estimate. A large OOF-minus-test gap suggests overfitting, while similarly weak scores on both splits suggest underfitting. The final cross-model interpretation belongs in Notebook 10, where all model families are scored on the same normalized scale.
